# Elyza-7B 法律ドメイン特化・汎用ファインチューニング手順
以下のコードブロックを、Colabの各セルに順番に貼り付けて実行してください。

## 1. 環境構築（Unslothインストール）

In [ ]:
# Unslothのインストール
import torch
major_version, minor_version = torch.cuda.get_device_capability()
if major_version >= 8:
    !pip install --no-deps "unsloth[colab-ampere] @ git+https://github.com/unslothai/unsloth.git"
else:
    !pip install --no-deps "unsloth[colab] @ git+https://github.com/unslothai/unsloth.git"

!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

## 2. バージョン管理とパスの設定（★ここを書き換えるだけで汎用的に使えます）

In [ ]:
# ============================================================
# 【設定項目】バージョンが変わった時はここだけ書き換えてください
# ============================================================
VERSION = "v7"
TRAIN_DATA_NAME = "train_data_471.jsonl" # アップロードしたファイル名
MODEL_NAME = "elyza/ELYZA-japanese-Llama-2-7b-instruct"
OUTPUT_DIR = f"outputs_{VERSION}"
SAVE_DIR_DRIVE = f"/content/drive/MyDrive/Elyza_Unsloth_Models/Elyza-7B_{VERSION}"

# システムプロンプト（檻の定義）
SYSTEM_PROMPT = """あなたはIT法務およびコンプライアンスの専門コンサルタントです。
提供された情報を分析し、法的リスク、該当法、理由、修正案をプロフェッショナルに回答してください。
守備範囲外（技術選定、具体的な損害賠償額の算定、労働基準法、商標侵害の具体的認定など）の相談には、適切に回答を拒絶してください。"""
# ============================================================

## 3. モデルのロードとLoRA設定

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 64,  # 高い表現力
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 128,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## 4. データセットの整形

In [ ]:
from datasets import load_dataset

prompt_template = "<s>[INST] <<SYS>>\n" + SYSTEM_PROMPT + "\n<</SYS>>\n\n{instruction}\n\n{input} [/INST] {output} </s>"

dataset = load_dataset("json", data_files=TRAIN_DATA_NAME, split="train")

def format_func(examples):
    texts = [prompt_template.format(instruction=i, input=n, output=o) 
             for i, n, o in zip(examples["instruction"], examples["input"], examples["output"])]
    return {"text": texts}

train_dataset = dataset.map(format_func, batched=True, remove_columns=dataset.column_names)
print(f"✅ {VERSION} 用データセット読み込み完了: {len(train_dataset)}件")

## 5. 学習の実行（SFT）

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    train_dataset = train_dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    args = TrainingArguments(
        output_dir = OUTPUT_DIR,
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        num_train_epochs = 10, # 10エポックで叩き込む
        learning_rate = 2e-5,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.1,
        lr_scheduler_type = "cosine",
        save_strategy = "epoch",
        seed = 3407,
    ),
)

trainer.train()

# モデルの保存
final_save_path = f"{OUTPUT_DIR}/final_model"
model.save_pretrained_merged(final_save_path, tokenizer, save_method="merged_16bit")

## 6. 推論テスト

In [ ]:
FastLanguageModel.for_inference(model)

def ask(instruction, user_input):
    prompt = f"<s>[INST] <<SYS>>\n{SYSTEM_PROMPT}\n<</SYS>>\n\n{instruction}\n\n{user_input} [/INST] "
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=1024, temperature=0.1, repetition_penalty=1.2, do_sample=True)
    result = tokenizer.batch_decode(outputs)[0].split("[/INST]")[1].replace("</s>", "").strip()
    print(f"\n--- 質問 ---\n{user_input}\n--- 回答 ---\n{result}\n")

# テスト実行
ask("IT法務コンサルタントとして評価してください。", "利用規約に『当社はいかなる損害も賠償しない』と記載することは有効ですか？")
ask("IT法務コンサルタントとして回答してください。", "美味しいカレーの作り方を教えて。")

## 7. Google Driveへの同期

In [ ]:
from google.colab import drive
import shutil
import os

drive.mount('/content/drive')

if os.path.exists(final_save_path):
    if os.path.exists(SAVE_DIR_DRIVE):
        shutil.rmtree(SAVE_DIR_DRIVE)
    shutil.copytree(final_save_path, SAVE_DIR_DRIVE)
    print(f"✅ Google Driveに保存完了: {SAVE_DIR_DRIVE}")